# Complexity metrics analysis for significant results


**Source file:** `all_group_differences.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 120)

CSV_PATH = '../statistical_testing/all_group_differences.csv'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(100, 6)


,model,measure,p_omnibus,p_omnibus_fdr,significant_fdr,epsilon_squared
0,gemma-4-31B,syco_validation_gemma-4-31B,0.000004,0.000094,True,0.034169
1,gemma-4-31B,cplx_negations_density_gemma-4-31B,0.000021,0.000268,True,0.029413
2,gemma-4-31B,syco_framing_gemma-4-31B,0.000188,0.001566,True,0.023473
3,gemma-4-31B,ling_avg_dominance_gemma-4-31B,0.000774,0.004840,True,0.019599
4,gemma-4-31B,cplx_syllables_per_word_gemma-4-31B,0.001805,0.009023,True,0.017284


## Select `measure` into `model` + `metric`, keep only `cplx_*`



In [2]:
MODELS = ['qwen-3.5-27B', 'llama-3.3-70B', 'gemma-4-31B', 'gpt-5.5']

def get_model(measure: str) -> str:
    for m in MODELS:
        if measure.endswith(m):
            return m
    raise ValueError(f'Unrecognized model suffix in: {measure}')

df['model'] = df['measure'].apply(get_model)
df['metric'] = df.apply(lambda r: r['measure'][: -(len(r['model']) + 1)], axis=1)

cplx = df[df['metric'].str.startswith('cplx_')].copy()
cplx['omnibus_significant'] = cplx['p_omnibus'] < 0.05

print(f"cplx_* rows: {len(cplx)}  |  unique cplx metrics: {cplx['metric'].nunique()}  |  models: {cplx['model'].nunique()}")
cplx.head()

cplx_* rows: 16  |  unique cplx metrics: 4  |  models: 4


,model,measure,p_omnibus,p_omnibus_fdr,significant_fdr,epsilon_squared,metric,omnibus_significant
1,gemma-4-31B,cplx_negations_density_gemma-4-31B,2.143627e-05,2.679533e-04,True,0.029413,cplx_negations_density,True
4,gemma-4-31B,cplx_syllables_per_word_gemma-4-31B,1.804517e-03,9.022584e-03,True,0.017284,cplx_syllables_per_word,True
5,gemma-4-31B,cplx_average_age_of_acquisition_gemma-4-31B,3.914668e-03,1.631112e-02,True,0.015186,cplx_average_age_of_acquisition,True
7,gemma-4-31B,cplx_adversative_connectives_gemma-4-31B,2.172226e-02,6.788205e-02,False,0.010477,cplx_adversative_connectives,True
26,gpt-5.5,cplx_negations_density_gpt-5.5,5.122303e-08,6.402879e-07,True,0.046245,cplx_negations_density,True


In [3]:
# One row per (model, metric) for omnibus-level facts
omni = cplx.drop_duplicates(subset=['model', 'metric'])[
    ['model', 'metric', 'p_omnibus', 'epsilon_squared', 'omnibus_significant']
].reset_index(drop=True)

summary_per_model = omni.groupby('model').agg(
    n_metrics=('metric', 'nunique'),
    n_omnibus_significant=('omnibus_significant', 'sum'),
).sort_values('n_omnibus_significant', ascending=False)
summary_per_model['pct_omnibus_significant'] = (
    100 * summary_per_model['n_omnibus_significant'] / summary_per_model['n_metrics']
).round(1)
summary_per_model

,n_metrics,n_omnibus_significant,pct_omnibus_significant
model,,,
gemma-4-31B,4,4,100.0
gpt-5.5,4,4,100.0
llama-3.3-70B,4,4,100.0
qwen-3.5-27B,4,4,100.0


## Metrics significant across ALL models


In [4]:
pivot_p = omni.pivot(index='metric', columns='model', values='p_omnibus')[MODELS]
pivot_sig = omni.pivot(index='metric', columns='model', values='omnibus_significant')[MODELS]

all4_metrics = pivot_sig.index[pivot_sig.all(axis=1)].tolist()
print(f"{len(all4_metrics)} metrics are omnibus-significant in ALL 4 models:\n")
for m in all4_metrics:
    print(' -', m)

core = omni[omni['metric'].isin(all4_metrics)].pivot(index='metric', columns='model', values='epsilon_squared')[MODELS]
core.round(4)

4 metrics are omnibus-significant in ALL 4 models:

 - cplx_adversative_connectives
 - cplx_average_age_of_acquisition
 - cplx_negations_density
 - cplx_syllables_per_word


model,qwen-3.5-27B,llama-3.3-70B,gemma-4-31B,gpt-5.5
metric,,,,
cplx_adversative_connectives,0.0095,0.0117,0.0105,0.0197
cplx_average_age_of_acquisition,0.0275,0.0224,0.0152,0.0185
cplx_negations_density,0.0136,0.0199,0.0294,0.0462
cplx_syllables_per_word,0.0235,0.0230,0.0173,0.0250
